# RAGAS Metrics — 20-Minute Live Demo

**Unit 3 companion demo — Module 1: "Evaluating RAG: Faithfulness & Groundedness"**

RAGAS (Retrieval-Augmented Generation Assessment) is the evaluation framework named on that
slide. Rather than importing it as a black box, this notebook **implements a simplified,
from-scratch version of its four core metrics** using Claude as the judge — so you can see
exactly what each metric is actually checking, before you ever call the real library.

| Time | Section |
|---|---|
| 0:00–2:00 | Setup + a deliberately mixed-quality evaluation dataset |
| 2:00–6:00 | Metric 1 — **Faithfulness** |
| 6:00–10:00 | Metric 2 — **Answer Relevancy** |
| 10:00–14:00 | Metric 3 — **Context Precision** |
| 14:00–17:00 | Metric 4 — **Context Recall** |
| 17:00–19:00 | Aggregate scorecard + chart |
| 19:00–20:00 | Wrap-up + the real `ragas` library |

> **Facilitator tip:** the three examples in the dataset are deliberately engineered to fail
> in three *different* ways — run all four metrics on all three examples and pause on the
> score table. The pattern of which metric catches which failure is the whole point.

**You'll need:** an Anthropic API key (same one from Demo 1 / Demo 2). No `ragas` package
install is required for the core demo — that comes up only in the optional wrap-up cell.


In [ ]:
# Setup (0:00–1:00)
!pip install -q anthropic sentence-transformers


In [ ]:
import os, getpass

try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    api_key = None

if not api_key:
    api_key = getpass.getpass("Enter your Anthropic API key: ")

os.environ["ANTHROPIC_API_KEY"] = api_key

import anthropic
client = anthropic.Anthropic()
JUDGE_MODEL = "claude-sonnet-4-5"
print("Client ready.")


## The Evaluation Dataset (1:00–2:00)

Same insurance-policy knowledge base used elsewhere in Unit 3, plus **three RAG outputs
already run for you** — each engineered to fail in a different way, so the metrics have
something interesting to disagree about:

1. **`good_example`** — correct retrieval, faithful answer. Everything should score well.
2. **`hallucinated_example`** — correct retrieval, but the generated answer adds a fact
   that isn't in the context. Faithfulness should catch this; context metrics should not.
3. **`bad_retrieval_example`** — the retriever pulled the wrong chunks entirely, and the
   answer confidently states a fabricated number as a result. Context metrics *and*
   faithfulness should both catch problems here.


In [ ]:
good_example = {
    "question": "What is the deductible for collision coverage?",
    "contexts": [
        "Collision Coverage: The Company will pay for direct and accidental physical loss "
        "to your covered auto caused by collision, subject to a $500 deductible per "
        "occurrence. Coverage applies regardless of fault.",
    ],
    "answer": "The deductible for collision coverage is $500 per occurrence.",
    "ground_truth": "The deductible for collision coverage is $500 per occurrence.",
}

hallucinated_example = {
    "question": "How much does the company pay for a rental car after a collision?",
    "contexts": [
        "Rental Reimbursement: If your covered auto is out of service due to a covered "
        "collision or comprehensive loss, the Company will reimburse rental costs up to "
        "$40 per day for a maximum of 30 days.",
    ],
    "answer": (
        "The company reimburses rental costs up to $40 per day for up to 30 days, and also "
        "provides a one-time $200 loyalty bonus for customers with more than 5 years of "
        "continuous coverage."
    ),  # the $200 loyalty bonus is fabricated — not in the context at all
    "ground_truth": "The company reimburses rental costs up to $40 per day for a maximum of 30 days.",
}

bad_retrieval_example = {
    "question": "What is the dollar threshold for escalating a claim to a Senior Claims Adjuster?",
    "contexts": [
        # Wrong chunks retrieved — the real escalation-threshold clause was never fetched
        "Water Damage Exclusion: Damage caused by flood, surface water, or sewer backup is "
        "excluded from standard coverage. Separate flood insurance must be purchased.",
        "Roadside Assistance: The Company will reimburse reasonable towing and labor costs "
        "up to $100 per disablement, limited to four disablements per policy period.",
    ],
    "answer": "Claims over $5,000 must be escalated to a Senior Claims Adjuster.",  # fabricated number
    "ground_truth": "Claims over $10,000 must be escalated to a Senior Claims Adjuster before settlement is authorized.",
}

examples = {
    "good_example": good_example,
    "hallucinated_example": hallucinated_example,
    "bad_retrieval_example": bad_retrieval_example,
}
print(f"Loaded {len(examples)} evaluation examples.")


---
## Metric 1 — Faithfulness (2:00–6:00)

**What it measures:** of all the factual claims in the generated answer, what fraction are
actually supported by the retrieved context? This is RAGAS's core hallucination detector.

**How real RAGAS computes it:** (1) decompose the answer into atomic claims using an LLM,
(2) ask the LLM to verify each claim against the context independently, (3)
`faithfulness = supported_claims / total_claims`.


In [ ]:
def decompose_into_claims(answer: str) -> list[str]:
    prompt = (
        "Break the following answer into a numbered list of atomic factual claims — each "
        "claim should be a single, independently checkable statement. Respond with ONLY the "
        "numbered list, one claim per line, no other text.\n\n"
        f"Answer: {answer}"
    )
    resp = client.messages.create(model=JUDGE_MODEL, max_tokens=300,
                                   messages=[{"role": "user", "content": prompt}])
    lines = resp.content[0].text.strip().split("\n")
    claims = [l.split(".", 1)[-1].strip() for l in lines if l.strip()]
    return claims


def claim_supported_by_context(claim: str, contexts: list[str]) -> bool:
    context_text = "\n\n".join(contexts)
    prompt = (
        "Given the CONTEXT and a CLAIM, respond with only one word: YES if the claim is "
        "directly supported by the context, or NO if it is not supported or not mentioned.\n\n"
        f"CONTEXT:\n{context_text}\n\nCLAIM: {claim}"
    )
    resp = client.messages.create(model=JUDGE_MODEL, max_tokens=5,
                                   messages=[{"role": "user", "content": prompt}])
    return resp.content[0].text.strip().upper().startswith("YES")


def faithfulness(example: dict) -> dict:
    claims = decompose_into_claims(example["answer"])
    verdicts = [claim_supported_by_context(c, example["contexts"]) for c in claims]
    score = sum(verdicts) / len(verdicts) if verdicts else 0.0
    return {"score": score, "claims": list(zip(claims, verdicts))}


# Run it on all three examples
for name, ex in examples.items():
    result = faithfulness(ex)
    print(f"=== {name} — Faithfulness: {result['score']:.2f} ===")
    for claim, supported in result["claims"]:
        print(f"  [{'OK' if supported else 'UNSUPPORTED'}] {claim}")
    print()


**Discussion point:** `good_example` should score close to 1.0, and both
`hallucinated_example` and `bad_retrieval_example` should show at least one
`UNSUPPORTED` claim — the fabricated loyalty bonus and the fabricated $5,000 threshold,
respectively. This is exactly the check from the "Wealth Management Advisor Tool" example
in the deck.


---
## Metric 2 — Answer Relevancy (6:00–10:00)

**What it measures:** does the answer actually address *this specific question* — as
opposed to being faithful-but-off-topic, or vague, or generic?

**How real RAGAS computes it:** generate several synthetic questions that the *answer*
would be a good response to, embed them, and compare their similarity to the *original*
question. A focused, on-topic answer produces synthetic questions that closely match the
original; a vague or generic answer produces synthetic questions that drift.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def generate_synthetic_questions(answer: str, n: int = 3) -> list[str]:
    prompt = (
        f"Generate {n} different questions that the following ANSWER would be a good, "
        "direct response to. Respond with ONLY the questions, one per line, no numbering.\n\n"
        f"ANSWER: {answer}"
    )
    resp = client.messages.create(model=JUDGE_MODEL, max_tokens=200,
                                   messages=[{"role": "user", "content": prompt}])
    return [q.strip() for q in resp.content[0].text.strip().split("\n") if q.strip()]


def answer_relevancy(example: dict, n: int = 3) -> dict:
    synthetic_questions = generate_synthetic_questions(example["answer"], n=n)
    all_texts = [example["question"]] + synthetic_questions
    embeddings = embedder.encode(all_texts, normalize_embeddings=True)
    original_emb, synthetic_embs = embeddings[0], embeddings[1:]
    similarities = synthetic_embs @ original_emb
    return {"score": float(np.mean(similarities)), "synthetic_questions": synthetic_questions,
            "similarities": similarities.tolist()}


for name, ex in examples.items():
    result = answer_relevancy(ex)
    print(f"=== {name} — Answer Relevancy: {result['score']:.2f} ===")
    print(f"  Original question: {ex['question']}")
    for q, sim in zip(result["synthetic_questions"], result["similarities"]):
        print(f"  [{sim:.2f}] Reverse-engineered: {q}")
    print()


**Discussion point:** notice that `hallucinated_example` can still score *reasonably well*
here — the answer does address the rental-reimbursement question, it just also contains an
unsupported extra claim. **This is the key lesson:** answer relevancy and faithfulness catch
different failure modes, and a RAG system needs both, not just one.


---
## Metric 3 — Context Precision (10:00–14:00)

**What it measures:** of the chunks the retriever actually returned, how many were relevant
— and were the relevant ones ranked near the top? This scores the **retrieval** step, not
the generation step.

**How real RAGAS computes it:** judge each retrieved chunk's relevance to the question, then
compute a rank-weighted precision (Average Precision): chunks relevant near the top count
more than the same chunk buried at rank 5.


In [ ]:
def chunk_relevant_to_question(chunk: str, question: str) -> bool:
    prompt = (
        "Given a QUESTION and a retrieved CONTEXT CHUNK, respond with only one word: YES if "
        "the chunk contains information that helps answer the question, or NO if it is "
        "irrelevant.\n\n"
        f"QUESTION: {question}\n\nCONTEXT CHUNK: {chunk}"
    )
    resp = client.messages.create(model=JUDGE_MODEL, max_tokens=5,
                                   messages=[{"role": "user", "content": prompt}])
    return resp.content[0].text.strip().upper().startswith("YES")


def context_precision(example: dict) -> dict:
    relevance = [chunk_relevant_to_question(c, example["question"]) for c in example["contexts"]]
    if not any(relevance):
        return {"score": 0.0, "relevance": relevance}

    # Average Precision: at each relevant position k, compute precision@k, then average
    precisions_at_k = []
    num_relevant_so_far = 0
    for k, is_relevant in enumerate(relevance, start=1):
        if is_relevant:
            num_relevant_so_far += 1
            precisions_at_k.append(num_relevant_so_far / k)
    score = sum(precisions_at_k) / sum(relevance)
    return {"score": score, "relevance": relevance}


for name, ex in examples.items():
    result = context_precision(ex)
    print(f"=== {name} — Context Precision: {result['score']:.2f} ===")
    for chunk, is_rel in zip(ex["contexts"], result["relevance"]):
        print(f"  [{'RELEVANT' if is_rel else 'NOT RELEVANT'}] {chunk[:70]}...")
    print()


**Discussion point:** `bad_retrieval_example` should score **0.0** here — neither retrieved
chunk actually addresses the claims-escalation question, even though the model still
confidently generated an answer. This is the retrieval-layer failure the "Naive RAG" and
"Hybrid Search" slides in Module 1 were about — context precision is the metric that would
have caught it automatically in a CI regression suite.


---
## Metric 4 — Context Recall (14:00–17:00)

**What it measures:** of everything in the (human-written) **ground-truth answer**, how much
of it is actually backed by the retrieved context? This catches *missing* retrieval, as
opposed to *irrelevant* retrieval (which is what Context Precision caught).

**How real RAGAS computes it:** decompose the ground-truth answer into statements, and check
whether each statement can be attributed to the retrieved context.


In [ ]:
def statement_attributable_to_context(statement: str, contexts: list[str]) -> bool:
    context_text = "\n\n".join(contexts)
    prompt = (
        "Given the CONTEXT and a STATEMENT from a reference answer, respond with only one "
        "word: YES if the context contains enough information to support this statement, or "
        "NO if it does not.\n\n"
        f"CONTEXT:\n{context_text}\n\nSTATEMENT: {statement}"
    )
    resp = client.messages.create(model=JUDGE_MODEL, max_tokens=5,
                                   messages=[{"role": "user", "content": prompt}])
    return resp.content[0].text.strip().upper().startswith("YES")


def context_recall(example: dict) -> dict:
    statements = decompose_into_claims(example["ground_truth"])  # reuse the same decomposer
    attributable = [statement_attributable_to_context(s, example["contexts"]) for s in statements]
    score = sum(attributable) / len(attributable) if attributable else 0.0
    return {"score": score, "statements": list(zip(statements, attributable))}


for name, ex in examples.items():
    result = context_recall(ex)
    print(f"=== {name} — Context Recall: {result['score']:.2f} ===")
    for stmt, attr in result["statements"]:
        print(f"  [{'COVERED' if attr else 'MISSING'}] {stmt}")
    print()


**Discussion point:** `bad_retrieval_example` should score **0.0** here too — the $10,000
threshold in the ground truth is simply nowhere in the two retrieved chunks. Context
Precision and Context Recall often move together when retrieval is *completely* wrong, but
they can diverge: a retriever can return 10 chunks that are all technically relevant
(high precision) while still missing the one specific fact the ground truth needs (low
recall) — worth drawing on a whiteboard if you have one.


---
## Aggregate Scorecard (17:00–19:00)

Pull all four metrics together into the kind of table a real regression suite would produce
per build.


In [ ]:
import pandas as pd

rows = []
for name, ex in examples.items():
    rows.append({
        "example": name,
        "faithfulness": faithfulness(ex)["score"],
        "answer_relevancy": answer_relevancy(ex)["score"],
        "context_precision": context_precision(ex)["score"],
        "context_recall": context_recall(ex)["score"],
    })

scorecard = pd.DataFrame(rows).set_index("example").round(2)
scorecard


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 4.5))
for i, (name, row) in enumerate(scorecard.iterrows()):
    ax.bar(x + i * width, [row[m] for m in metrics], width, label=name)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics, rotation=15, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("RAGAS-style metrics across three example RAG outputs")
ax.legend()
plt.tight_layout()
plt.show()


---
## Wrap-Up (19:00–20:00)

**One line per metric, for the recap:**
- **Faithfulness** — is every claim in the answer backed by the retrieved context?
- **Answer Relevancy** — does the answer actually address the question that was asked?
- **Context Precision** — of what was retrieved, how much was relevant, and was it ranked well?
- **Context Recall** — of what the correct answer needs, how much did retrieval actually surface?

**Why all four, and not just one:** the three examples in this notebook were built so that
each failure mode lights up a *different* metric. A production regression suite (the "200-
question RAGAS suite" from the deck's Wealth Management example) runs all four together on
every pipeline change, because a change can improve one metric while quietly breaking
another.

### Using the Real `ragas` Library
This notebook's functions are simplified, teaching versions. The real `ragas` package
implements the same four metrics (plus several more) with more robust prompting, batching,
and support for many LLM/embedding providers. A minimal real-library equivalent looks like
this (not run here — it requires its own setup and is not needed for this demo):

```python
# pip install ragas datasets
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# from datasets import Dataset
#
# dataset = Dataset.from_list([good_example, hallucinated_example, bad_retrieval_example])
# results = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
```

The extension tasks below ask you to apply these same four metrics to new, different
real-life scenarios — some of which are designed to make you question whether these four
metrics are even enough.


---
---
# Extension Tasks — Different Real-Life Scenarios

Each task gives you a small, realistic (but invented) evaluation example from a different
industry. For each one: **(a)** predict which metric(s) will catch the problem before you
run the code, **(b)** run all four metrics, **(c)** check your prediction.

## Task 1 — Healthcare Triage Assistant (High-Stakes Faithfulness)
A symptom-checker's context correctly says "over-the-counter pain relievers may be used for
mild fever," but the generated answer specifies "take 1000mg of acetaminophen every 4
hours" — a specific dosage never mentioned in the context, and one that could be
medically incorrect for some patients.

```python
healthcare_example = {
    "question": "What can I take for a mild fever?",
    "contexts": [
        "For mild fever, over-the-counter pain relievers may be used as directed on the "
        "packaging. Consult a doctor if fever persists beyond 3 days or exceeds 103°F."
    ],
    "answer": "You can take 1000mg of acetaminophen every 4 hours for your fever.",
    "ground_truth": "Over-the-counter pain relievers may be used for mild fever, following "
                     "package directions; consult a doctor if it persists beyond 3 days.",
}
# Your turn: run all four metrics. Which one(s) catch the fabricated dosage?
# Why is this failure mode especially dangerous in a Tier 3 (high-risk) healthcare use case
# per the Module 2 risk-tiering framework?
```

## Task 2 — Financial Advisory Chatbot (Context Precision Noise)
A wealth-management assistant retrieves one genuinely relevant chunk about index funds, plus
two irrelevant chunks about an unrelated insurance product that happened to share vocabulary
("fund," "return") with the question.

```python
finance_example = {
    "question": "What are the fees on the S&P 500 index fund?",
    "contexts": [
        "The S&P 500 Index Fund carries an annual expense ratio of 0.04%, with no front-end "
        "or back-end sales load.",
        "The Variable Annuity Growth Fund guarantees a minimum death benefit return of "
        "principal, subject to a 7-year surrender period.",
        "Our Balanced Retirement Fund targets a mix of 60% equities and 40% fixed income "
        "for investors nearing retirement age.",
    ],
    "answer": "The S&P 500 index fund has an annual expense ratio of 0.04% with no sales load.",
    "ground_truth": "The S&P 500 index fund has an annual expense ratio of 0.04%, with no "
                     "front-end or back-end sales load.",
}
# Your turn: run context_precision(). With only 1 of 3 chunks relevant, what score do you
# expect? Does the answer still end up faithful despite the retrieval noise? What does that
# tell you about why context precision and faithfulness need to be tracked separately?
```

## Task 3 — Legal Contract Q&A (Multi-Hop Context Recall Gap)
A contract-review assistant retrieves the indemnification clause, but the real answer also
requires a cross-referenced liability cap defined in a *different* section — which
retrieval never fetched.

```python
legal_example = {
    "question": "What is the maximum indemnification liability under this agreement?",
    "contexts": [
        "Section 8.1 (Indemnification): Vendor agrees to indemnify Client against all "
        "third-party claims arising from Vendor's breach of this Agreement.",
    ],
    "answer": "Vendor must indemnify the Client for third-party claims arising from a breach, "
              "with no stated limit in the retrieved text.",
    "ground_truth": "Vendor must indemnify the Client for third-party claims arising from a "
                     "breach, capped at the total fees paid in the preceding 12 months per "
                     "Section 12.4 (Limitation of Liability).",
}
# Your turn: run context_recall(). Which statement in the ground truth gets marked MISSING?
# This is the exact "multi-hop retrieval" gap from Module 1 — what design pattern
# (prompt chaining, agentic multi-hop RAG, etc.) would you add to close it?
```

## Task 4 — HR Policy Bot After an Update (Regression Testing)
The same question, asked before and after a policy document changes. This models the
"semantic caching must invalidate on document update" lesson from Module 1.

```python
hr_before_update = {
    "question": "How many weeks of parental leave am I entitled to?",
    "contexts": ["Parental Leave Policy (v1, effective Jan 2024): Employees are entitled to "
                 "12 weeks of paid parental leave."],
    "answer": "You are entitled to 12 weeks of paid parental leave.",
    "ground_truth": "Employees are entitled to 12 weeks of paid parental leave.",
}

hr_after_update_stale_answer = {
    "question": "How many weeks of parental leave am I entitled to?",
    "contexts": ["Parental Leave Policy (v2, effective Jul 2024): Employees are entitled to "
                 "16 weeks of paid parental leave."],
    "answer": "You are entitled to 12 weeks of paid parental leave.",  # stale cached answer!
    "ground_truth": "Employees are entitled to 16 weeks of paid parental leave.",
}
# Your turn: run faithfulness() on hr_after_update_stale_answer. It should score LOW, because
# "12 weeks" is not supported by the v2 context. Explain in one sentence why a nightly
# automated RAGAS run — not just spot-checking — is what would catch this kind of staleness
# before an employee ever sees the wrong number.
```

## Task 5 — Customer Support Macro Bot (Low Answer Relevancy)
The retrieval and faithfulness are both fine — the bot just gives a generic canned response
instead of engaging with the specific question asked.

```python
support_example = {
    "question": "Why was I charged twice for my last order?",
    "contexts": [
        "Billing discrepancies can occur due to pre-authorization holds that are later "
        "released, or due to split shipments being billed separately. Contact support with "
        "your order ID for a specific review."
    ],
    "answer": "Thank you for reaching out! We're here to help with any questions you may "
              "have about your account or orders.",  # generic, doesn't actually answer
    "ground_truth": "Duplicate charges are often pre-authorization holds that get released, "
                     "or a split shipment billed in two parts; support can review your "
                     "specific order ID.",
}
# Your turn: this answer might score fine on faithfulness (it doesn't contradict anything —
# it just doesn't say anything). Which metric is designed specifically to catch this? Run it
# and see if your prediction holds.
```

## Task 6 — Bring Your Own Use Case (Open-Ended)
Using the use case from your Hands-On Lab architecture diagram (or risk-tiering worksheet),
construct your own 3-example evaluation set — one good, one faithfulness failure, one
retrieval failure — in the same dict format used throughout this notebook. Run all four
metrics. Which failure mode do you think is most likely for *your specific* use case, and
which metric would you therefore watch most closely in production monitoring (tying back to
the Module 3 "Monitoring, Drift & Incident Response" slide)?
